# STA masked mean-pooling smoke test

This notebook exercises the production Model 1 implementation against the validated eight-event feature shard. It checks masked pooling, output shapes, all-false masks and gradient flow.

## 1. Locate the project and load the contract shard

In [ ]:
import sys
from pathlib import Path

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "src" / "sta_features.py").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not locate the project root.")

sys.path.insert(0, str(PROJECT_ROOT))

import torch
from torch.nn import functional as F

from scripts.create_sta_example_shard import build_example_shard
from src.sta_features import load_sta_feature_shard, save_sta_feature_shard
from src.sta_model_1 import MaskedMeanPoolSTA

SHARD_PATH = (
    PROJECT_ROOT
    / "results"
    / "local_sta_contract"
    / "train"
    / "shard_00000.pt"
)

if not SHARD_PATH.exists():
    save_sta_feature_shard(build_example_shard(), SHARD_PATH)

shard = load_sta_feature_shard(SHARD_PATH, expected_split="train")
features = shard["features"].float()
valid_mask = shard["valid_mask"]

print(f"Project root: {PROJECT_ROOT}")
print(f"Shard: {SHARD_PATH}")
print("Features:", features.shape, features.dtype)
print("Mask:", valid_mask.shape, valid_mask.dtype)
print("Statuses:", shard["status"])

## 2. Load the production Model 1 implementation

`MaskedMeanPoolSTA` is imported from `src/sta_model_1.py`, so every check below exercises the same implementation used by the training pipeline. The validity mask removes padded temporal segments from the mean, and clamping the valid count to one makes an all-false event produce a zero pooled representation instead of a division by zero.

In [ ]:
print(f"Using production model: {MaskedMeanPoolSTA.__module__}.{MaskedMeanPoolSTA.__name__}")

## 3. Check the forward pass and output contract

In [ ]:
torch.manual_seed(42)
model = MaskedMeanPoolSTA()
model.eval()

with torch.inference_mode():
    verb_logits, noun_logits = model(features, valid_mask)

assert verb_logits.shape == (len(features), 97)
assert noun_logits.shape == (len(features), 300)
assert torch.isfinite(verb_logits).all()
assert torch.isfinite(noun_logits).all()

print("Verb logits:", verb_logits.shape)
print("Noun logits:", noun_logits.shape)
print("Forward-pass checks passed.")

## 4. Verify the masked mean directly

In [ ]:
pooled = model.pool_features(features, valid_mask)

for event_index in range(len(features)):
    selected = features[event_index][valid_mask[event_index]]
    expected = (
        selected.mean(dim=0)
        if len(selected)
        else torch.zeros_like(pooled[event_index])
    )
    assert torch.allclose(pooled[event_index], expected, atol=1e-6)

all_false_rows = ~valid_mask.any(dim=1)
assert all_false_rows.any(), "The contract shard should include no-context rows."
assert torch.all(pooled[all_false_rows] == 0)

print("Full, partial and all-false pooling checks passed.")

## 5. Prove that invalid segments cannot affect predictions

This deliberately fills padded positions with very large random values. In evaluation mode the outputs must remain unchanged because those positions are masked out.

In [ ]:
torch.manual_seed(43)
modified_features = features.clone()
modified_features[~valid_mask] = (
    torch.randn_like(modified_features[~valid_mask]) * 1_000_000
)

model.eval()
with torch.inference_mode():
    original_outputs = model(features, valid_mask)
    modified_outputs = model(modified_features, valid_mask)

assert torch.allclose(original_outputs[0], modified_outputs[0], atol=1e-6)
assert torch.allclose(original_outputs[1], modified_outputs[1], atol=1e-6)

print("Invalid segments do not affect predictions.")

## 6. Verify one training step can backpropagate

All-false events are excluded from the loss. They will later receive Model 0 prior probabilities during evaluation.

In [ ]:
model.train()
model.zero_grad(set_to_none=True)

has_context = valid_mask.any(dim=1)
verb_logits, noun_logits = model(features, valid_mask)
loss = (
    F.cross_entropy(
        verb_logits[has_context],
        shard["verb_labels"][has_context],
    )
    + F.cross_entropy(
        noun_logits[has_context],
        shard["noun_labels"][has_context],
    )
)

loss.backward()

for name, parameter in model.named_parameters():
    assert parameter.grad is not None, f"No gradient for {name}"
    assert torch.isfinite(parameter.grad).all(), f"Invalid gradient for {name}"

print(f"Loss: {loss.item():.4f}")
print("Backward-pass checks passed.")

## Next step

The production implementation and permanent unit tests are now in `src/sta_model_1.py` and `tests/test_sta_model_1.py`. The next model task is the masked GRU prototype.